# Stamina

**Stamina** measures how well a pitcher maintains their **stuff** — velocity and spin rate — as they accumulate pitches, both *within a single outing* (long-haul endurance) and *within a single inning* (high-stress grit).

Each component is the OLS slope of (value / unit-baseline) regressed against pitch index, computed per (outing × pitch_type) or (inning × pitch_type), then usage-weighted to a per-pitcher number. A less-negative slope = better stamina. Stamina+ is normalised so league average = 100.

## Notebook structure

| Phase | What runs | Re-run when... |
|---|---|---|
| **0 — Config** | Thresholds, weights, normalization method | Any parameter change |
| **1 — Data pull** | Fetch pitch-by-pitch from Baseball Savant | New season / first run |
| **2 — Metric computation** | Compute per-pitcher slopes from parquet | Threshold / qualifier changes |
| **3 — Scoring** | Apply weights, shrinkage, normalization | Weight / normalization changes |
| **4 — Analysis** | YoY repeatability, component heatmap, quadrant scatter, external validation | Any time results change |


---
## Metric Definitions

Four components, all measured as **OLS slopes** of (value / baseline) vs pitch index. Sign convention: more-negative = larger decline.

| Component | Unit | Stuff | Baseline | Interpretation |
|---|---|---|---|---|
| `velo_slope_outing` | one outing (≥20 pitches) | `release_speed` | first 5 pitches of each pitch_type in the outing | velocity drop per pitch over the outing |
| `spin_slope_outing` | one outing (≥20 pitches) | `release_spin_rate` | first 5 of each pitch_type | spin drop per pitch over the outing |
| `velo_slope_inning` | one inning (≥15 pitches) | `release_speed` | first 5 of each pitch_type in the inning | velocity drop per pitch within a stressful inning |
| `spin_slope_inning` | one inning (≥15 pitches) | `release_spin_rate` | first 5 of each pitch_type | spin drop per pitch within the inning |

Slopes from each (unit × pitch_type) cell are pitch-count-weighted across all the pitcher's outings/innings of that pitch type, then usage-weighted across pitch types into one per-pitcher number per component.

### Sub-scores
- **Outing Stamina** — `velo_slope_outing` + `spin_slope_outing` (long-haul endurance)
- **Inning Stamina** — `velo_slope_inning` + `spin_slope_inning` (within-inning grit)


---
## Phase 0 — Config & Setup

In [ ]:
%pip install pybaseball

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from tqdm import tqdm
import pybaseball
from pybaseball import statcast_single_game, playerid_reverse_lookup, pitching_stats
import warnings
warnings.filterwarnings('ignore')
pybaseball.cache.enable()

import subprocess
_git_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], cwd=os.path.abspath(".")).decode().strip()
_nb_dir = os.path.join(_git_root, "stamina")
DATA_DIR = os.path.join(_nb_dir, "data")
RESULTS_DIR = os.path.join(_nb_dir, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEASONS = [2021, 2022, 2023, 2024, 2025]

# ── Qualifiers (re-run Phase 2 if you change these) ──────────────────────────
MIN_PITCHES        = 500   # season minimum to appear on leaderboard
MIN_OUTING_PITCHES = 20    # outing must hit this to contribute to outing slopes
MIN_INNING_PITCHES = 15    # inning must hit this to contribute to inning slopes
MIN_TYPE_IN_UNIT   = 5     # ≥5 pitches of a given pitch_type in a unit to compute slope
BASELINE_PITCHES   = 5     # first N pitches of each pitch_type set the unit baseline

# ── Per-component weights (re-run Phase 3 if you change these) ───────────────
# Default: equal weight within each sub-score. Set 0 to exclude.
# Note: lowerIsBetter = False for all four (higher / less-negative slope = better),
# so Phase 3 does NOT flip signs after normalising.
WEIGHTS = {
    'velo_slope_outing': 1.0,
    'spin_slope_outing': 1.0,
    'velo_slope_inning': 1.0,
    'spin_slope_inning': 1.0,
}
METRIC_COLS = list(WEIGHTS.keys())

# ── Sub-score groupings (re-run Phase 3 if you change these) ─────────────────
# Outing: long-haul endurance across a full outing
OUTING_METRICS = ['velo_slope_outing', 'spin_slope_outing']
# Inning: short-burst grit within high-stress innings
INNING_METRICS = ['velo_slope_inning', 'spin_slope_inning']
OUTING_WEIGHT  = 0.5
INNING_WEIGHT  = 0.5

# ── Normalization method (re-run Phase 3 if you change this) ─────────────────
NORMALIZATION = 'zscore'   # 'zscore' | 'percentile' | 'minmax'

# ── Sample-size shrinkage (re-run Phase 3 if you change this) ────────────────
# Slope estimates are already aggregated across many pitches, so a light prior
# is enough. Same value for outing and inning since both come from many cells.
PRIOR_OBS_OUTING = 250
PRIOR_OBS_INNING = 250

# ── Score confidence scaling (re-run Phase 3 if you change this) ─────────────
SCORE_SCALE      = 40
CONFIDENCE_PRIOR = 1000

print(f'Config loaded. Data dir: {DATA_DIR}')
print(f'Normalization: {NORMALIZATION} | MIN_PITCHES: {MIN_PITCHES}')
print(f'Qualifiers — outing ≥{MIN_OUTING_PITCHES} pitches | inning ≥{MIN_INNING_PITCHES} pitches | '
      f'≥{MIN_TYPE_IN_UNIT} of pitch_type per unit')

---
## Phase 1 — Data Pull

Bulk-fetches Statcast pitch data per season via `pybaseball.statcast(start_dt, end_dt)`, broken into **monthly chunks** so each chunk's progress is visible and Ctrl-C is non-destructive (completed chunks land in `data/_chunks_YYYY/` and are skipped on re-run). Filters to regular season (`game_type == 'R'`) and saves the stitched output to `data/all_pitches_YYYY.parquet`.

Stamina needs `release_speed`, `release_spin_rate`, `inning`, `inning_topbot` in addition to the standard PA columns.

In [ ]:
# ── Shared fetch helpers ──────────────────────────────────────────────────────

KEEP_COLS = [
    'game_pk', 'game_date', 'pitcher',
    'at_bat_number', 'pitch_number',
    'inning', 'inning_topbot',
    'pitch_type', 'pitch_name', 'description', 'events',
    'release_speed', 'release_spin_rate',
    'estimated_woba_using_speedangle',   # used in Phase 4E for late-PA wOBA delta
]
NUMERIC_COLS = [
    'at_bat_number', 'pitch_number', 'game_pk', 'inning',
    'release_speed', 'release_spin_rate', 'estimated_woba_using_speedangle',
]

# Generous bounds around any modern MLB regular season — pybaseball returns
# nothing for off-season dates and game_type filtering drops Spring Training / postseason.
SEASON_BOUNDS = {
    yr: (f'{yr}-03-15', f'{yr}-10-15') for yr in range(2018, 2030)
}

def _normalize_raw(df):
    df = df[[c for c in KEEP_COLS if c in df.columns]].copy()
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    return df


def _week_chunks(start_dt, end_dt):
    """Yield (chunk_start, chunk_end) pairs spanning [start_dt, end_dt] in ~weekly chunks.

    Smaller chunks = smaller CSV payloads, less chance Savant returns an HTML
    error page mid-response, and finer-grained progress + retries.
    """
    cur = pd.to_datetime(start_dt)
    end = pd.to_datetime(end_dt)
    while cur <= end:
        chunk_end = min(cur + pd.Timedelta(days=6), end)
        yield cur.strftime('%Y-%m-%d'), chunk_end.strftime('%Y-%m-%d')
        cur = chunk_end + pd.Timedelta(days=1)


def _fetch_chunk(c_start, c_end, retries=3, sleep_s=4):
    """Fetch one date-range chunk from Savant, retrying on parser/network errors."""
    import time
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            raw = pybaseball.statcast(start_dt=c_start, end_dt=c_end, verbose=False)
            if 'game_type' in raw.columns:
                raw = raw[raw['game_type'] == 'R']
            return _normalize_raw(raw)
        except Exception as e:
            last_err = e
            print(f'    attempt {attempt}/{retries} failed: {type(e).__name__}: {e}', flush=True)
            if attempt < retries:
                time.sleep(sleep_s * attempt)   # 4s, 8s, 12s back-off
    raise RuntimeError(f'Chunk {c_start}→{c_end} failed after {retries} retries: {last_err}')


def pull_season(season):
    """Load pitch data from parquet cache; else fetch week-by-week with per-chunk progress.

    Weekly chunks give visible progress every ~10-30s and let you Ctrl-C without
    losing completed chunks — they land in `data/_chunks_YYYY/` and are skipped
    on re-run. Failed chunks are retried with back-off before surfacing the error.
    """
    path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'{season}: loaded {len(df):,} pitches from cache.')
        return df

    start_dt, end_dt = SEASON_BOUNDS[season]
    chunk_dir = os.path.join(DATA_DIR, f'_chunks_{season}')
    os.makedirs(chunk_dir, exist_ok=True)

    chunks = list(_week_chunks(start_dt, end_dt))
    print(f'{season}: {len(chunks)} weekly chunks {start_dt} → {end_dt}')

    for c_start, c_end in chunks:
        c_path = os.path.join(chunk_dir, f'{c_start}_to_{c_end}.parquet')
        if os.path.exists(c_path):
            print(f'  ✓ {c_start} → {c_end}  (cached)')
            continue
        print(f'  · {c_start} → {c_end}  fetching...', flush=True)
        norm = _fetch_chunk(c_start, c_end)
        norm.to_parquet(c_path, index=False)
        print(f'    saved {len(norm):,} pitches', flush=True)

    chunk_files = sorted(os.path.join(chunk_dir, f) for f in os.listdir(chunk_dir)
                         if f.endswith('.parquet'))
    df = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
    df.to_parquet(path, index=False)
    print(f'{season}: {len(df):,} regular-season pitches saved → {path}')
    return df


print('Fetch helpers defined.')

In [ ]:
# ── 2021 data pull ──────────────────────────────────────────────────────────
pitches_2021 = pull_season(2021)

In [ ]:
# ── 2022 data pull ──────────────────────────────────────────────────────────
pitches_2022 = pull_season(2022)

In [ ]:
# ── 2023 data pull ──────────────────────────────────────────────────────────
pitches_2023 = pull_season(2023)

In [ ]:
# ── 2024 data pull ──────────────────────────────────────────────────────────
pitches_2024 = pull_season(2024)

In [ ]:
# ── 2025 data pull ──────────────────────────────────────────────────────────
pitches_2025 = pull_season(2025)

---
## Phase 2 — Metric Computation

For each (unit × pitch_type) cell with ≥`MIN_TYPE_IN_UNIT` pitches of that type, compute the closed-form OLS slope of (value / baseline) vs pitch index. Aggregate up to per-pitcher with pitch-count weighting. Save paired `_n` columns so Phase 3 can apply Laplace shrinkage.

In [ ]:
# ── Shared metric helpers ─────────────────────────────────────────────────────

def _prep(pitches):
    """Sort pitches and add pidx_outing / pidx_inning indices plus unit-size columns."""
    p = pitches.sort_values(['game_pk', 'pitcher', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
    p['pidx_outing'] = p.groupby(['game_pk', 'pitcher']).cumcount() + 1
    p['pidx_inning'] = p.groupby(['game_pk', 'pitcher', 'inning', 'inning_topbot']).cumcount() + 1
    p['outing_pitches'] = p.groupby(['game_pk', 'pitcher'])['pitch_number'].transform('size')
    p['inning_pitches'] = p.groupby(['game_pk', 'pitcher', 'inning', 'inning_topbot'])['pitch_number'].transform('size')
    return p


def _unit_slopes(df, unit_cols, pidx_col, value_col):
    """One closed-form OLS slope per (unit × pitch_type), regressing value/baseline on pidx.

    Returns DataFrame indexed/columned: [pitcher, slope, n] — collapsed across the
    unit and pitch_type dimensions but with one row per cell so the caller can
    pitch-weight-aggregate to a per-pitcher number.

    Skips cells with: <MIN_TYPE_IN_UNIT pitches of the type, baseline ≤ 0, or
    zero variance in pidx (degenerate regression).
    """
    group_cols = unit_cols + ['pitch_type']
    d = df[df[value_col].notna() & df['pitch_type'].notna()].copy()
    if d.empty:
        return pd.DataFrame(columns=['pitcher', 'slope', 'n'])

    # Order so cumcount picks the chronologically first pitches as the baseline window.
    d = d.sort_values(group_cols + [pidx_col]).reset_index(drop=True)
    d['type_rank']  = d.groupby(group_cols).cumcount() + 1
    d['type_size']  = d.groupby(group_cols)[pidx_col].transform('size')
    d = d[d['type_size'] >= MIN_TYPE_IN_UNIT]
    if d.empty:
        return pd.DataFrame(columns=['pitcher', 'slope', 'n'])

    baseline = (
        d[d['type_rank'] <= BASELINE_PITCHES]
        .groupby(group_cols)[value_col].mean()
        .rename('baseline')
        .reset_index()
    )
    d = d.merge(baseline, on=group_cols, how='inner')
    d = d[d['baseline'] > 0]
    d['pct'] = d[value_col] / d['baseline']

    d['x']  = d[pidx_col].astype(float)
    d['y']  = d['pct'].astype(float)
    d['xy'] = d['x'] * d['y']
    d['xx'] = d['x'] * d['x']

    agg = d.groupby(group_cols).agg(
        n=('x', 'size'),
        sx=('x', 'sum'),
        sy=('y', 'sum'),
        sxy=('xy', 'sum'),
        sxx=('xx', 'sum'),
    )
    denom = agg['n'] * agg['sxx'] - agg['sx'] ** 2
    slope = np.where(denom > 0, (agg['n'] * agg['sxy'] - agg['sx'] * agg['sy']) / denom, np.nan)
    agg['slope'] = slope
    return agg.reset_index()[['pitcher', 'slope', 'n']].dropna(subset=['slope'])


def _aggregate(cells):
    """Pitch-weighted average of cell-level slopes per pitcher.

    Returns two Series indexed by pitcher: (weighted_slope, total_n).
    """
    if cells.empty:
        return pd.Series(dtype=float, name='slope'), pd.Series(dtype=float, name='n')
    cells = cells.copy()
    cells['wn'] = cells['slope'] * cells['n']
    g = cells.groupby('pitcher')
    total_n = g['n'].sum()
    weighted = g['wn'].sum() / total_n
    return weighted.rename('slope'), total_n.rename('n')


def compute_metrics(season):
    pitches_path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if not os.path.exists(pitches_path):
        raise FileNotFoundError(f'Run the Phase 1 pull cell for {season} first.')

    print(f'{season}: loading pitches...')
    pitches = pd.read_parquet(pitches_path)
    pitches = _prep(pitches)

    out_eligible = pitches[pitches['outing_pitches'] >= MIN_OUTING_PITCHES]
    inn_eligible = pitches[pitches['inning_pitches'] >= MIN_INNING_PITCHES]
    print(f'  {len(out_eligible):,} pitches in qualifying outings | '
          f'{len(inn_eligible):,} pitches in qualifying innings')

    print(f'{season}: computing slopes (outing × velo / spin, inning × velo / spin)...')
    vso_cells = _unit_slopes(out_eligible, ['game_pk', 'pitcher'],
                             'pidx_outing', 'release_speed')
    sso_cells = _unit_slopes(out_eligible, ['game_pk', 'pitcher'],
                             'pidx_outing', 'release_spin_rate')
    vsi_cells = _unit_slopes(inn_eligible, ['game_pk', 'pitcher', 'inning', 'inning_topbot'],
                             'pidx_inning', 'release_speed')
    ssi_cells = _unit_slopes(inn_eligible, ['game_pk', 'pitcher', 'inning', 'inning_topbot'],
                             'pidx_inning', 'release_spin_rate')

    vso, vso_n = _aggregate(vso_cells)
    sso, sso_n = _aggregate(sso_cells)
    vsi, vsi_n = _aggregate(vsi_cells)
    ssi, ssi_n = _aggregate(ssi_cells)

    pitch_count = pitches.groupby('pitcher').size().rename('pitch_count')

    metrics = pd.DataFrame({
        'velo_slope_outing':   vso,
        'velo_slope_outing_n': vso_n,
        'spin_slope_outing':   sso,
        'spin_slope_outing_n': sso_n,
        'velo_slope_inning':   vsi,
        'velo_slope_inning_n': vsi_n,
        'spin_slope_inning':   ssi,
        'spin_slope_inning_n': ssi_n,
    }).join(pitch_count, how='outer')

    # Pitcher index is MLBAM int — ensure that's preserved.
    metrics.index.name = 'pitcher'

    out_path = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    metrics.to_parquet(out_path)
    print(f'{season}: {len(metrics)} pitchers saved → {out_path}')
    return metrics

print('Metric helpers defined.')

In [ ]:
# ── 2021 metric computation ─────────────────────────────────────────────────
metrics_2021 = compute_metrics(2021)

In [ ]:
# ── 2022 metric computation ─────────────────────────────────────────────────
metrics_2022 = compute_metrics(2022)

In [ ]:
# ── 2023 metric computation ─────────────────────────────────────────────────
metrics_2023 = compute_metrics(2023)

In [ ]:
# ── 2024 metric computation ─────────────────────────────────────────────────
metrics_2024 = compute_metrics(2024)

In [ ]:
# ── 2025 metric computation ─────────────────────────────────────────────────
metrics_2025 = compute_metrics(2025)

---
## Phase 3 — Scoring

Loads `pitcher_metrics_YYYY.parquet` for all seasons. Applies Laplace shrinkage (lighter prior than composure since each slope is already aggregated across many pitches), normalises within season, weights into the two sub-scores, combines into a composite, and converts to a confidence-weighted `stamina_plus` where league average = 100 by construction.

**Sign convention:** higher / less-negative slope = better stamina, so normalised values are **not** flipped (unlike composure, where lower walk-rate = better required flipping).

In [ ]:
def _normalize(series, method):
    s = series.copy().astype(float)
    if method == 'zscore':
        return (s - s.mean()) / s.std()
    if method == 'percentile':
        return s.rank(pct=True, na_option='keep')
    if method == 'minmax':
        return (s - s.min()) / (s.max() - s.min())
    raise ValueError(f'Unknown normalization: {method}')


def score_season(season):
    path = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Run the Phase 2 compute cell for {season} first.')

    df = pd.read_parquet(path)
    df = df[df['pitch_count'] >= MIN_PITCHES].copy()

    active_out = [m for m in OUTING_METRICS if WEIGHTS.get(m, 0) > 0]
    active_inn = [m for m in INNING_METRICS if WEIGHTS.get(m, 0) > 0]
    active = active_out + active_inn

    # ── Laplace shrinkage on the slope itself ─────────────────────────────────
    # shrunk = (n · slope + PRIOR · league_avg_slope) / (n + PRIOR)
    # Volume-weighted league avg, pitchers with n=0 stay NaN and are skipped later.
    for col in active:
        n_col = col + '_n'
        if n_col not in df.columns:
            continue
        n = df[n_col].fillna(0)
        total_n = n.sum()
        league_avg = (df[col] * n).sum() / total_n if total_n > 0 else df[col].mean()
        prior = PRIOR_OBS_OUTING if col in OUTING_METRICS else PRIOR_OBS_INNING
        df[col] = np.where(
            n > 0,
            (n * df[col] + prior * league_avg) / (n + prior),
            np.nan,
        )

    # ── Normalize & weight (no sign flip — higher slope = better stamina) ─────
    normed = pd.DataFrame(index=df.index)
    for col in active:
        normed[col] = _normalize(df[col], NORMALIZATION) * WEIGHTS[col]

    # ── Outing sub-score ──────────────────────────────────────────────────────
    out_w  = [WEIGHTS[c] for c in active_out]
    out_ws = normed[active_out].notna().multiply(out_w).sum(axis=1)
    df['outing_score'] = normed[active_out].sum(axis=1, skipna=True) / out_ws

    # ── Inning sub-score ──────────────────────────────────────────────────────
    inn_w  = [WEIGHTS[c] for c in active_inn]
    inn_ws = normed[active_inn].notna().multiply(inn_w).sum(axis=1)
    df['inning_score'] = normed[active_inn].sum(axis=1, skipna=True) / inn_ws

    # ── Composite stamina — weighted avg of sub-scores ────────────────────────
    o_valid = df['outing_score'].notna().astype(float)
    i_valid = df['inning_score'].notna().astype(float)
    df['stamina_score'] = (
        df['outing_score'].fillna(0) * OUTING_WEIGHT * o_valid +
        df['inning_score'].fillna(0) * INNING_WEIGHT * i_valid
    ) / (OUTING_WEIGHT * o_valid + INNING_WEIGHT * i_valid)

    def to_plus(series, pitch_counts):
        mu, sigma = series.mean(), series.std()
        deviation = (series - mu) / sigma * SCORE_SCALE
        confidence = pitch_counts / (pitch_counts + CONFIDENCE_PRIOR)
        weighted_dev = deviation * confidence
        weighted_dev = weighted_dev - weighted_dev.mean()   # anchor mean to 0 → plus = 100
        return (100 + weighted_dev).round(1)

    df['stamina_plus']        = to_plus(df['stamina_score'], df['pitch_count'])
    df['outing_stamina_plus'] = to_plus(df['outing_score'],  df['pitch_count'])
    df['inning_stamina_plus'] = to_plus(df['inning_score'],  df['pitch_count'])

    lookup = playerid_reverse_lookup(df.index.tolist(), key_type='mlbam').set_index('key_mlbam')
    df['pitcher_name'] = df.index.map(lookup['name_last'] + ', ' + lookup['name_first'])
    df['season'] = season
    df['pitcher_id'] = df.index   # MLBAM int — needed for external joins

    final = (
        df[[
            'pitcher_id', 'season', 'pitcher_name', 'pitch_count',
            'stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus',
            'stamina_score', 'outing_score', 'inning_score',
        ] + active]
        .dropna(subset=['stamina_score'])
        .sort_values('stamina_plus', ascending=False)
        .reset_index(drop=True)
    )

    out = os.path.join(RESULTS_DIR, f'stamina_scores_{season}.csv')
    final.to_csv(out, index=False)
    print(f'{season}: {len(final)} qualified pitchers → {out}')
    return final


pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

all_scores = {}
for yr in SEASONS:
    if not os.path.exists(os.path.join(DATA_DIR, f'pitcher_metrics_{yr}.parquet')):
        print(f'{yr}: no metrics file yet — run Phase 2 first.')
        continue
    all_scores[yr] = score_season(yr)

for yr, scores in all_scores.items():
    print(f'\n=== {yr} — Top 15 stamina ===')
    display(scores[[
        'pitcher_name', 'pitch_count',
        'stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus',
    ]].head(15))
    print(f'\n=== {yr} — Bottom 15 stamina ===')
    display(scores[[
        'pitcher_name', 'pitch_count',
        'stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus',
    ]].tail(15))

---
## Multi-year view

Loads saved CSVs to compare stamina across seasons. No re-computation needed.

In [ ]:
csvs = [
    os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv')
    for yr in SEASONS
    if os.path.exists(os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv'))
]
all_years = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)

print('=== Multi-season pitchers (avg Stamina+, min 2 seasons) ===')
multi = (
    all_years.groupby('pitcher_name')
    .agg(
        seasons=('season', 'count'),
        avg_stamina_plus=('stamina_plus', 'mean'),
        avg_outing_plus=('outing_stamina_plus', 'mean'),
        avg_inning_plus=('inning_stamina_plus', 'mean'),
    )
    .query('seasons > 1')
    .sort_values('avg_stamina_plus', ascending=False)
)
display(multi.head(30))

---
## Phase 4 — Analysis

Diagnostic and validation cells on Phase 3 results.

### Cell A — Score distributions & sub-score relationship

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

csvs = [os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv')
        for yr in SEASONS
        if os.path.exists(os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv'))]

if not csvs:
    print('No scored seasons found — run Phase 3 first.')
    all_df, seasons_present = pd.DataFrame(), []
else:
    all_df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    seasons_present = sorted(all_df['season'].unique())

    fig, ax = plt.subplots(figsize=(7, 6))
    colors = cm.tab10.colors
    for idx, yr in enumerate(seasons_present):
        sub = all_df[all_df['season'] == yr].dropna(
            subset=['outing_stamina_plus', 'inning_stamina_plus'])
        if sub.empty:
            continue
        r = sub[['outing_stamina_plus', 'inning_stamina_plus']].corr().iloc[0, 1]
        ax.scatter(sub['outing_stamina_plus'], sub['inning_stamina_plus'],
                   alpha=0.45, s=18, color=colors[idx % 10], label=f'{yr}  r={r:.2f}')

    overall = all_df.dropna(subset=['outing_stamina_plus', 'inning_stamina_plus'])
    r_all = overall[['outing_stamina_plus', 'inning_stamina_plus']].corr().iloc[0, 1]
    ax.axline((100, 100), slope=1, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel('Outing Stamina+')
    ax.set_ylabel('Inning Stamina+')
    ax.set_title(f'Outing vs Inning Stamina+  (overall r = {r_all:.2f})')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f'Overall Pearson r = {r_all:.3f}  (n={len(overall)} pitcher-seasons)')

### Cell B — Year-over-year repeatability

Stamina should be a real, persistent trait — target r > 0.4 for both sub-scores.

In [ ]:
# Depends on all_df and seasons_present from Cell A.
yoy_rows = []
for yr in seasons_present[:-1]:
    cols = ['pitcher_name', 'stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus']
    t0 = all_df[all_df['season'] == yr][cols].dropna()
    t1 = all_df[all_df['season'] == yr + 1][cols].dropna()
    merged = t0.merge(t1, on='pitcher_name', suffixes=('_t0', '_t1'))
    if len(merged) < 10:
        continue
    for col in ['stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus']:
        r = merged[[f'{col}_t0', f'{col}_t1']].corr().iloc[0, 1]
        yoy_rows.append({'seasons': f'{yr}→{yr+1}', 'metric': col, 'r': round(r, 3), 'n': len(merged)})

if yoy_rows:
    yoy = pd.DataFrame(yoy_rows).pivot(index='seasons', columns='metric', values='r')
    yoy.columns.name = None
    print('Year-over-year Pearson r  (> 0.40 = strongly repeatable | < 0.20 = mostly noise)\n')
    display(yoy)
else:
    print('Need at least 2 scored consecutive seasons for year-over-year analysis.')

### Cell C — Component correlation heatmap

In [ ]:
# Depends on seasons_present from Cell A.
metric_dfs = []
for yr in seasons_present:
    path = os.path.join(DATA_DIR, f'pitcher_metrics_{yr}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        df = df[df['pitch_count'] >= MIN_PITCHES]
        metric_dfs.append(df[METRIC_COLS])

if not metric_dfs:
    print('No pitcher_metrics parquets found — run Phase 2 first.')
else:
    combined = pd.concat(metric_dfs, ignore_index=True).dropna(how='all')
    corr = combined.corr()

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks(range(len(METRIC_COLS)))
    ax.set_yticks(range(len(METRIC_COLS)))
    ax.set_xticklabels(METRIC_COLS, rotation=35, ha='right', fontsize=8)
    ax.set_yticklabels(METRIC_COLS, fontsize=8)
    for i in range(len(METRIC_COLS)):
        for j in range(len(METRIC_COLS)):
            ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                    fontsize=8, color='black')
    n_out = len(OUTING_METRICS)
    for block_start, block_size in [(0, n_out), (n_out, len(INNING_METRICS))]:
        rect = plt.Rectangle((block_start - 0.5, block_start - 0.5),
                             block_size, block_size,
                             fill=False, edgecolor='black', lw=2)
        ax.add_patch(rect)
    ax.set_title('Component slope correlations\n(boxes = outing block / inning block)')
    plt.tight_layout()
    plt.show()

### Cell D — Divergent pitcher quadrants

High outing stamina + low inning stamina = a workhorse who fades within stressful innings. The reverse profile is a pitcher who recovers within an inning but burns out over the start.

In [ ]:
# Depends on all_df from Cell A.
DIVERGENCE_THRESHOLD = 10  # stamina+ points

div = all_df.copy().dropna(subset=['outing_stamina_plus', 'inning_stamina_plus'])
div['divergence'] = div['outing_stamina_plus'] - div['inning_stamina_plus']
show_cols = ['season', 'pitcher_name', 'pitch_count',
             'stamina_plus', 'outing_stamina_plus', 'inning_stamina_plus', 'divergence']

print('=== High outing / low inning  (holds stuff across the start; fades in stressful innings) ===')
display(
    div[div['divergence'] > DIVERGENCE_THRESHOLD][show_cols]
    .sort_values('divergence', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print('\n=== High inning / low outing  (rebounds within an inning; loses stuff over the full outing) ===')
display(
    div[div['divergence'] < -DIVERGENCE_THRESHOLD][show_cols]
    .sort_values('divergence')
    .head(15)
    .reset_index(drop=True)
)

### Cell E — External validation

Three checks per the spec:
1. **Innings pitched / workload** — primary face-validity signal (high stamina ↑ → more IP)
2. **Late-PA wOBA delta** — wOBA on PAs deep in the outing (pidx_outing ≥ 60) minus early (≤ 30); high stamina should mean smaller delta (negative correlation)
3. **ERA / FIP** — weak negative prior; stamina is one ingredient of value

In [ ]:
# ── External validation: Stamina+ vs IP / ERA / FIP / late-PA wOBA delta ─────
csvs = [os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv')
        for yr in SEASONS
        if os.path.exists(os.path.join(RESULTS_DIR, f'stamina_scores_{yr}.csv'))]

if not csvs:
    print('No scored seasons found — run Phase 3 first.')
else:
    stam_df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    scored_seasons = sorted(stam_df['season'].unique())
    print(f'Loaded stamina scores: {len(stam_df)} pitcher-seasons across {scored_seasons}')
    stam_df['pitcher_id'] = pd.to_numeric(stam_df['pitcher_id'], errors='coerce')

    def parse_ip(ip_str):
        """'6.2' → 6⅔ innings. Decimal part is outs (0/1/2), not tenths."""
        parts = str(ip_str).split('.')
        full = int(parts[0])
        outs = int(parts[1]) if len(parts) > 1 else 0
        return full + outs / 3.0

    def fetch_pitching(season):
        url = (f'https://statsapi.mlb.com/api/v1/stats'
               f'?stats=season&group=pitching&season={season}'
               f'&sportId=1&gameType=R&playerPool=All&limit=2000')
        splits = requests.get(url).json()['stats'][0]['splits']
        rows = []
        for s in splits:
            st  = s['stat']
            ip  = parse_ip(st.get('inningsPitched', 0))
            bb  = int(st.get('baseOnBalls',    0))
            hbp = int(st.get('hitByPitch',     0))
            hr  = int(st.get('homeRuns',       0))
            k   = int(st.get('strikeOuts',     0))
            er  = int(st.get('earnedRuns',     0))
            bf  = int(st.get('battersFaced',   0))
            rows.append({
                'pitcher_id': s['player']['id'],
                'ERA':   (er * 9 / ip) if ip > 0 else None,
                'BB_pct': bb / bf      if bf > 0 else None,
                'FIP_num': (13*hr + 3*(bb + hbp) - 2*k),
                'IP':    ip,
                'season': season,
            })
        df = pd.DataFrame(rows)
        lg_ip  = df['IP'].sum()
        lg_er  = sum(int(s['stat'].get('earnedRuns', 0)) for s in splits)
        lg_num = df['FIP_num'].sum()
        fip_const = (lg_er * 9 / lg_ip) - (lg_num / lg_ip) if lg_ip > 0 else 3.15
        df['FIP'] = np.where(df['IP'] > 0, df['FIP_num'] / df['IP'] + fip_const, None)
        return df.drop(columns=['FIP_num'])

    stat_frames = []
    for yr in scored_seasons:
        try:
            stat_frames.append(fetch_pitching(yr))
            print(f'  {yr}: fetched')
        except Exception as e:
            print(f'  {yr}: FAILED — {e}')
    stat_all = pd.concat(stat_frames, ignore_index=True) if stat_frames else pd.DataFrame()

    merged = stam_df.copy()
    if not stat_all.empty:
        stat_all['pitcher_id'] = pd.to_numeric(stat_all['pitcher_id'], errors='coerce')
        merged = merged.merge(stat_all, on=['pitcher_id', 'season'], how='inner')
        print(f'After IP/ERA/FIP merge: {len(merged)} pitcher-seasons')

    for col in ['ERA', 'BB_pct', 'FIP', 'IP']:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors='coerce')

In [ ]:
# ── Late-PA wOBA delta (computed directly from parquet) ──────────────────────
# For each (pitcher, season): mean estimated_woba_using_speedangle on PAs ending
# with pidx_outing >= 60 (late) minus PAs ending with pidx_outing <= 30 (early).
# Positive delta = pitcher gives up more contact damage late in outings.

late_rows = []
for yr in scored_seasons:
    path = os.path.join(DATA_DIR, f'all_pitches_{yr}.parquet')
    if not os.path.exists(path):
        continue
    p = pd.read_parquet(path)
    p['pidx_outing'] = (
        p.sort_values(['game_pk', 'pitcher', 'at_bat_number', 'pitch_number'])
         .groupby(['game_pk', 'pitcher']).cumcount() + 1
    )
    # Keep only PA-final pitches with a non-null xwOBA (i.e. contact PAs).
    pa_final = p.dropna(subset=['estimated_woba_using_speedangle']).copy()
    pa_final['estimated_woba_using_speedangle'] = pd.to_numeric(
        pa_final['estimated_woba_using_speedangle'], errors='coerce')
    early = pa_final[pa_final['pidx_outing'] <= 30].groupby('pitcher')['estimated_woba_using_speedangle']
    late  = pa_final[pa_final['pidx_outing'] >= 60].groupby('pitcher')['estimated_woba_using_speedangle']

    df = pd.DataFrame({
        'pitcher_id':       early.mean().index.union(late.mean().index),
    })
    df = df.set_index('pitcher_id')
    df['early_xwoba'] = early.mean()
    df['late_xwoba']  = late.mean()
    df['early_n']     = early.count()
    df['late_n']      = late.count()
    df['late_xwoba_delta'] = df['late_xwoba'] - df['early_xwoba']
    df = df[(df['early_n'] >= 25) & (df['late_n'] >= 25)]   # require sample on both sides
    df['season'] = yr
    df = df.reset_index()
    late_rows.append(df[['pitcher_id', 'season', 'late_xwoba_delta', 'early_n', 'late_n']])

if late_rows:
    late_df = pd.concat(late_rows, ignore_index=True)
    late_df['pitcher_id'] = pd.to_numeric(late_df['pitcher_id'], errors='coerce')
    merged = merged.merge(late_df, on=['pitcher_id', 'season'], how='left')
    print(f'Pitchers with computable late-PA wOBA delta: '
          f'{merged["late_xwoba_delta"].notna().sum()} / {len(merged)}')
else:
    print('No parquets available for late-PA wOBA delta.')

In [ ]:
# ── Scatter plots: Stamina+ vs IP, late-PA wOBA delta, ERA, FIP ──────────────
if 'merged' not in dir() or merged.empty:
    print('Run the merge cells above first.')
else:
    comparisons = [
        ('IP',                'IP (higher = more workload)',         '+'),
        ('late_xwoba_delta',  'late-early xwOBA delta (lower = better)', '−'),
        ('ERA',               'ERA (lower = better)',                '−'),
        ('FIP',               'FIP (lower = better)',                '−'),
    ]
    comparisons = [(c, lbl, exp) for c, lbl, exp in comparisons if c in merged.columns]
    if not comparisons:
        print('No external metric columns available.')
    else:
        fig, axes = plt.subplots(1, len(comparisons), figsize=(5.5 * len(comparisons), 5))
        if len(comparisons) == 1:
            axes = [axes]
        fig.suptitle('Stamina+ vs External Metrics', fontsize=13, y=1.02)
        all_seasons_u = sorted(merged['season'].unique())
        season_colors = {yr: plt.cm.tab10.colors[i % 10] for i, yr in enumerate(all_seasons_u)}

        for ax, (col, xlabel, expected) in zip(axes, comparisons):
            sub = merged[['stamina_plus', col, 'season']].dropna()
            if sub.empty:
                ax.set_title(f'{col}: no data')
                continue
            x, y = sub[col], sub['stamina_plus']
            r = sub[[col, 'stamina_plus']].corr().iloc[0, 1]
            for yr in all_seasons_u:
                mask = sub['season'] == yr
                if mask.any():
                    ax.scatter(x[mask], y[mask], alpha=0.35, s=16,
                               color=season_colors[yr], label=str(yr))
            m, b = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, m * x_line + b, color='black', lw=1.2, ls='--')
            ax.axhline(100, color='grey', lw=0.6, ls=':', label='League avg')
            ax.set_xlabel(xlabel, fontsize=9)
            if ax is axes[0]:
                ax.set_ylabel('Stamina+', fontsize=9)
            ax.set_title(f'r = {r:+.3f}  (n={len(sub)}, expected {expected})', fontsize=10)
            ax.legend(fontsize=7, markerscale=1.5, loc='best')

        plt.tight_layout()
        plt.show()

        print('\nCorrelation with Stamina+:')
        print(f'  {"Metric":<32} {"r":>7}  {"n":>6}  expected')
        print(f'  {"-"*55}')
        for col, label, exp in comparisons:
            sub = merged[['stamina_plus', col]].dropna()
            r = sub.corr().iloc[0, 1]
            print(f'  {label:<32} {r:>+7.3f}  {len(sub):>6}   {exp}')